In [2]:
import numpy as np

def objective_function(x):
    return -x**2 + 5 * x + 20

def pso_maximize(func, particles, c1=1, c2=1, u1=0.213, u2=0.876, w=0, iterations=3):
    n_particles = len(particles)
    x = np.array(particles, dtype=float)
    v = np.zeros(n_particles)  # Initial velocity = 0 (inertia w = 0)
    
    
    fitness = func(x)
    p_best = x.copy()
    p_best_fitness = fitness.copy()
    
    g_best_idx = np.argmax(p_best_fitness)
    g_best = p_best[g_best_idx]
    
    print(f"=== Initial Setup (t=0) ===")
    print(f"Particles (x): {x}")
    print(f"Fitness: {fitness}")
    print(f"Initial g_best: {g_best} (Fitness: {p_best_fitness[g_best_idx]:.4f})\n")
    
        for t in range(1, iterations + 1):
        
        v = w * v + c1 * u1 * (p_best - x) + c2 * u2 * (g_best - x)
        
        x = x + v
        
        
        fitness = func(x)
        
        
        improved = fitness > p_best_fitness
        p_best[improved] = x[improved]
        p_best_fitness[improved] = fitness[improved]
        
        
        g_best_idx = np.argmax(p_best_fitness)
        g_best = p_best[g_best_idx]
        
       
        print(f"=== Iteration {t} ===")
        print(f"Velocities (V): {np.round(v, 4)}")
        print(f"Positions (x): {np.round(x, 4)}")
        print(f"Fitness: {np.round(fitness, 4)}")
        print(f"Personal Bests (P_best): {np.round(p_best, 4)}")
        print(f"Global Best (g_best): {g_best:.4f} (Fitness: {p_best_fitness[g_best_idx]:.4f})\n")


initial_particles = [-9.6, -6.0, -2.6, -1.1, 0.6, 0.3, 2.8, 8.3, 10.0]
pso_maximize(
    func=objective_function, 
    particles=initial_particles, 
    c1=1, c2=1,          # Acceleration coefficients[cite: 3]
    u1=0.213, u2=0.876,  # Random numbers[cite: 3]
    w=0,                 # Inertia weight[cite: 3]
    iterations=3         # Number of iterations to run
)

=== Initial Setup (t=0) ===
Particles (x): [-9.6 -6.  -2.6 -1.1  0.6  0.3  2.8  8.3 10. ]
Fitness: [-120.16  -46.      0.24   13.29   22.64   21.41   26.16   -7.39  -30.  ]
Initial g_best: 2.8 (Fitness: 26.1600)

=== Iteration 1 ===
Velocities (V): [10.8624  7.7088  4.7304  3.4164  1.9272  2.19    0.     -4.818  -6.3072]
Positions (x): [1.2624 1.7088 2.1304 2.3164 2.5272 2.49   2.8    3.482  3.6928]
Fitness: [24.7183 25.624  26.1134 26.2163 26.2493 26.2499 26.16   25.2857 24.8272]
Personal Bests (P_best): [1.2624 1.7088 2.1304 2.3164 2.5272 2.49   2.8    3.482  3.6928]
Global Best (g_best): 2.4900 (Fitness: 26.2499)

=== Iteration 2 ===
Velocities (V): [ 1.0754  0.6843  0.315   0.1521 -0.0326  0.     -0.2716 -0.869  -1.0537]
Positions (x): [2.3378 2.3931 2.4454 2.4685 2.4946 2.49   2.5284 2.613  2.6391]
Fitness: [26.2237 26.2386 26.247  26.249  26.25   26.2499 26.2492 26.2372 26.2306]
Personal Bests (P_best): [2.3378 2.3931 2.4454 2.4685 2.4946 2.49   2.5284 2.613  2.6391]
Global Best 

In [5]:
import numpy as np

cities = ['A', 'B', 'C', 'D', 'E']
n = len(cities)

d = np.array([
    [0,  2,  9, 10,  7],  # A
    [2,  0,  6,  4,  3],  # B
    [9,  6,  0,  8,  5],  # C
    [10, 4,  8,  0,  5],  # D
    [7,  3,  5,  5,  0]   # E
], dtype=float)


alpha = 1.0        # Pheromone weight
beta = 2.0         # Visibility weight
psi = 0.5          # Evaporation constant (rho)
Q = 100.0          # Pheromone deposit constant
tau = np.ones((n, n))  # Initial pheromone matrix = 1


eta = np.zeros((n, n))
for i in range(n):
    for j in range(n):
        if i != j:
            eta[i, j] = 1.0 / d[i, j]


def calculate_probabilities(current_node, visited, tau, eta, alpha, beta):
    probabilities = np.zeros(n)
    denom = 0.0
    allowed = [j for j in range(n) if j not in visited]
    
    for j in allowed:
        val = (tau[current_node, j] ** alpha) * (eta[current_node, j] ** beta)
        probabilities[j] = val
        denom += val
        
    if denom > 0:
        probabilities /= denom
    return probabilities


def construct_tour(start_node, tau, eta, alpha, beta, d):
    tour = [start_node]
    visited = {start_node}
    
    while len(visited) < n:
        curr = tour[-1]
        probs = calculate_probabilities(curr, visited, tau, eta, alpha, beta)
        next_node = np.argmax(probs)  
        tour.append(next_node)
        visited.add(next_node)
        
    tour.append(start_node)  
    
    
    length = 0.0
    for i in range(len(tour) - 1):
        length += d[tour[i], tour[i+1]]
    return tour, length


start_nodes = [0, 1, 2]
tours = []
lengths = []

print("--- Ant Tour Construction ---")
for s in start_nodes:
    tour, length = construct_tour(s, tau, eta, alpha, beta, d)
    tours.append(tour)
    lengths.append(length)
    tour_names = [cities[i] for i in tour]
    print(f"Ant starting at {cities[s]}: {' -> '.join(tour_names)} | Length = {length}")


tau_new = (1 - psi) * tau

delta_tau = np.zeros((n, n))
for tour, length in zip(tours, lengths):
    deposit = Q / length
    for i in range(len(tour) - 1):
        u, v = tour[i], tour[i+1]
        delta_tau[u, v] += deposit
        delta_tau[v, u] += deposit 
tau_final = tau_new + delta_tau

print("\n--- Final Updated Pheromone Matrix ---")
print(f"{'':<6}" + "".join([f"{c:>10}" for c in cities]))
for i, row in enumerate(tau_final):
    row_str = "".join([f"{val:>10.4f}" for val in row])
    print(f"{cities[i]:<6}" + row_str)

--- Ant Tour Construction ---
Ant starting at A: A -> B -> E -> C -> D -> A | Length = 28.0
Ant starting at B: B -> A -> E -> C -> D -> B | Length = 26.0
Ant starting at C: C -> E -> B -> A -> D -> C | Length = 28.0

--- Final Updated Pheromone Matrix ---
               A         B         C         D         E
A         0.5000   11.4890    0.5000    7.6429    4.3462
B        11.4890    0.5000    0.5000    4.3462    7.6429
C         0.5000    0.5000    0.5000   11.4890   11.4890
D         7.6429    4.3462   11.4890    0.5000    0.5000
E         4.3462    7.6429   11.4890    0.5000    0.5000
